In [24]:
import pandas as pd
import numpy as np
df = pd.read_csv('fmnist_small.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0


In [25]:
from sklearn.model_selection import train_test_split
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [26]:
X_train = X_train/255
X_test = X_test/255

In [27]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [28]:
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.long)
y_test = torch.from_numpy(y_test).to(torch.long)

In [29]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [30]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [31]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )

  def forward(self, num_features):
    return self.model(num_features)

In [32]:
learning_rate = 0.1
epochs = 100

In [33]:
model = Neural_Network(X_train.shape[1])
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:
    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:0.7438684701919556
Epoch: 2, Loss:0.6104503273963928
Epoch: 3, Loss:0.6893749237060547
Epoch: 4, Loss:0.5306816697120667
Epoch: 5, Loss:0.449408620595932
Epoch: 6, Loss:0.66437828540802
Epoch: 7, Loss:0.4483320116996765
Epoch: 8, Loss:0.3609032928943634
Epoch: 9, Loss:0.6456291079521179
Epoch: 10, Loss:0.3507789969444275
Epoch: 11, Loss:0.2953684628009796
Epoch: 12, Loss:0.336178183555603
Epoch: 13, Loss:0.5447565913200378
Epoch: 14, Loss:0.23903559148311615
Epoch: 15, Loss:0.30897608399391174
Epoch: 16, Loss:0.45252418518066406
Epoch: 17, Loss:0.4191034436225891
Epoch: 18, Loss:0.10162702947854996
Epoch: 19, Loss:0.4031762480735779
Epoch: 20, Loss:0.1701325625181198
Epoch: 21, Loss:0.3963436484336853
Epoch: 22, Loss:0.20673868060112
Epoch: 23, Loss:0.2158694863319397
Epoch: 24, Loss:0.32624557614326477
Epoch: 25, Loss:0.17013829946517944
Epoch: 26, Loss:0.5649510025978088
Epoch: 27, Loss:0.40549394488334656
Epoch: 28, Loss:0.13718092441558838
Epoch: 29, Loss:0.327560663

In [36]:
model.eval()
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    y_pred = model(batch_features)
    _, predicted = torch.max(y_pred, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
print(f'Accuracy: {correct/total}')

Accuracy: 0.8516666666666667
